Excel-xml-json-zip-files processing
https://www.mssqltips.com/sqlservertip/7180/pyspark-etl-code-excel-xml-json-zip-files-azure-databricks/

In [0]:
excel=spark.read.format("com.crealytics.spark.excel").option("header", "true").option("sheetName", "Orders").option("dataAddress", "'Orders'!A1:F600").option("inferSchema", "true").option("useHeader", "true").option("treatEmptyValuesAsNulls", "true").option("addColorColumns", "False").load("/mnt/raw/Orders.xlsx")

display(excel)

In [0]:
csv=spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/raw/dimdates.csv")
display(csv)

In [0]:
csv.select("*").write.format('com.crealytics.spark.excel').option("header","true").option("inferSchema","true").save('/mnt/raw/dimdate.xlsx',mode="overwrite")


In [0]:
csv_to_xls=spark.read.format("com.crealytics.spark.excel").option("header", "true").option("inferSchema", "true").load("/mnt/raw/dimdate.xlsx")

In [0]:
import xlsxwriter
from shutil import copyfile
 
workbook = xlsxwriter.Workbook("/mnt/raw/Orders.xlsx")
worksheet = workbook.add_worksheet()
worksheet.write(0, 0, "Key")
worksheet.write(0, 1, "Value")
workbook.close()
 
copyfile('/mnt/raw/Orders.xlsx', '/mnt/raw/dimdate.xlsx')

xml files
https://learn.microsoft.com/en-us/previous-versions/windows/desktop/ms762271(v=vs.85)
https://learn.microsoft.com/en-us/dotnet/standard/linq/sample-xml-file-multiple-purchase-orders

In [0]:
df=spark.read.text("/mnt/raw/books.xml")
display(df)

In [0]:
df=spark.read.format("com.databricks.spark.xml").option("rootTag", "Catalog").option("rowTag","book").load("/mnt/raw/books.xml")
display(df)

In [0]:
df.select("*").write.format('com.databricks.spark.xml').option("rootTag", "Catalog").option("rowTag","book").save('/mnt/raw/booksnew.xml',mode="overwrite")
df=spark.read.format("com.databricks.spark.xml").option("rootTag", "Catalog").option("rowTag","book").load("/mnt/raw/booksnew.xml")
display(df)

In [0]:
%sql
CREATE TABLE books
USING xml
OPTIONS (path "/mnt/raw/books.xml", rowTag "book");
SELECT * FROM books;

In [0]:
json=spark.read.json(&apos;/mnt/raw/Customer1.json&apos;)
display(json)

In [0]:
/ For reference, here are the first three rows of the Customer1 file to shown the structure of the json format.


{"customerid":1,"Title":"Mr.","FirstName":"Orlando","MiddleName":"N.","LastName":"Gee"}
{"customerid":2,"Title":"Mr.","FirstName":"Keith","LastName":"Harris"}
{"customerid":3,"Title":"Ms.","FirstName":"Donna","MiddleName":"F.","LastName":"Carreras"}

In [0]:
%sql 
CREATE TEMPORARY VIEW json_table
USING json
OPTIONS (path "/mnt/raw/Customer1.json")
 
%sql 
SELECT * FROM json_table
WHERE customerid>5

In [0]:
dbutils.fs.put("/mnt/raw/multiline.json", """
[
    {"string":"string1","int":1,"array":[0,1,2],"key/value": {"key": "value1"}},
    {"string":"string2","int":2,"array":[3,4,5],"key/value": {"key": "value2"}},
    {"string":"string2","int":2,"array":[6,7,8],"key/value": {"key": "value2"}}
]""",True)

multiline_json=spark.read.option(&apos;multiline&apos;,"true").json("/mnt/raw/multiline.json")
display(multiline_json)

In [0]:
# RDD[String] storing a JSON object per string
data = ['{"booktitle":"The Azure Data Lakehouse Toolkit","author":{"firstname":"Ron","lastname":"LEsteve"}}']
rdd = sc.parallelize(data)
df = spark.read.json(rdd)
df.printSchema()

In [0]:
display(df.select("booktitle","author.firstname","author.lastname"))

In [0]:
df.select("booktitle","author.firstname","author.lastname").show()

ZIP FILES

In [0]:
%python
#Code to import the urllib package and read the contents of the specified URL zip file.
import urllib 
urllib.request.urlretrieve("https://resources.lendingclub.com/LoanStats3a.csv.zip", "/tmp/LoanStats3a.csv.zip")
 
 
#Code the unzip the url zip file
%sh
unzip /tmp/LoanStats3a.csv.zip
tail -n +2 LoanStats3a.csv > temp.csv
rm LoanStats3a.csv
 
 
#Code to list the files in the folder
%fs ls file:/databricks/driver/

In [0]:
%python
# Infer schema
df = (
    spark.read.format("xml")
    .option("rowTag", "book")
    .load("dbfs:/books.xml")
)

selected_data = df.select("author", "_id")

(
    selected_data.write.format("xml")
    .option("rootTag", "books")
    .option("rowTag", "book")
    .save("dbfs:/newbooks.xml")
)


In [0]:
SELECT
  book:bookid.author,     
  book:bookid.title,  
  book:bookid.genre,  
FROM book;

In [0]:
/* function to return max amount of books sold
** for the specified book_id
*/
CREATE FUNCTION udf_max_copies_sold_for_title (@book_id CHAR(6))
RETURNS INT
AS
BEGIN
   DECLARE @qty INT
   -- initialize the variable at 0:
   SELECT @qty = 0
 
   SELECT 
      @qty = MAX(qty) 
   FROM sales 
   WHERE
      book_id = @book_id
 
   /* If there are no books sold for book_id specified 
   ** then return 0:
   */
   RETURN ISNULL(@qty, 0)
END